### TDSP · Fase 3: Modelado con clases rebalanceadas

Segunda ejecución: los mismos modelos, con dos formas de compensar el desbalance de clases.

- **Entrada:** `Sample_Data/For_Modeling/train_clean.csv`
- **Salida:** `Code/Modeling/results/cv_balanced.csv`
- **Variable objetivo:** `Revenue`

#### Librerías

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

#### Semilla

In [3]:
# Semilla para reproducibilidad. Puede modificarse o eliminarse.
SEED = 1
np.random.seed(SEED)

#### Rutas

Relativas a la carpeta del notebook.

In [4]:
FOR_MODELING_DIR = Path("../../Sample_Data/For_Modeling")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [5]:
train = pd.read_csv(FOR_MODELING_DIR / "train_clean.csv")

TARGET = "Revenue"
X = train.drop(columns=["row_id", TARGET])
y = train[TARGET]

print(f"Sesiones: {len(X):,} | Compras: {y.sum():,} ({y.mean():.1%})")

Sesiones: 9,864 | Compras: 1,526 (15.5%)


#### Conjuntos de variables

Cada modelo se entrena con todas las variables y sin `PageValues`.

In [6]:
CATEGORICAL = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
NUMERIC_ALL = [c for c in X.columns if c not in CATEGORICAL]

FEATURE_SETS = {
    "con_PageValues": NUMERIC_ALL,
    "sin_PageValues": [c for c in NUMERIC_ALL if c != "PageValues"],
}

#### Preprocesador

Las variables numéricas pasan sin cambios, porque los árboles no dependen de la escala. Las categóricas se codifican en one-hot; los códigos con menos de 30 sesiones se agrupan en una sola columna.

In [7]:
def make_preprocessor(numeric):
    return ColumnTransformer(
        [
            ("num", "passthrough", numeric),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30,
                                  sparse_output=False), CATEGORICAL),
        ],
        verbose_feature_names_out=False,
    )

#### Modelos y estrategias de balanceo

- **Sobremuestreo:** `RandomOverSampler` duplica sesiones con compra hasta igualar ambas clases. Se aplica solo dentro de cada pliegue de entrenamiento.
- **Pesos de clase:** `class_weight="balanced"` hace que cada error en la clase "compra" cuente más, sin modificar los datos.

In [8]:
MODELS = {
    "arbol_sobremuestreo": (
        [("sampler", RandomOverSampler(random_state=SEED)),
         ("model", DecisionTreeClassifier(random_state=SEED))],
        {"model__max_depth": [3, 5, 8, None], "model__min_samples_leaf": [1, 5, 20]},
    ),
    "rf_sobremuestreo": (
        [("sampler", RandomOverSampler(random_state=SEED)),
         ("model", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))],
        {"model__max_depth": [10, 20, None], "model__min_samples_leaf": [1, 5]},
    ),
    "rf_pesos_de_clase": (
        [("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                           random_state=SEED, n_jobs=-1))],
        {"model__max_depth": [10, 20, None], "model__min_samples_leaf": [1, 5]},
    ),
}

#### Validación cruzada

Cinco pliegues estratificados. Para cada modelo, `GridSearchCV` elige los parámetros con mayor F1 de la clase "compra".

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {
    "f1": make_scorer(f1_score, zero_division=0),
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "average_precision": "average_precision",
}

rows = []
for set_name, numeric in FEATURE_SETS.items():
    for model_name, (steps, grid) in MODELS.items():
        pipe = Pipeline([("preprocessor", make_preprocessor(numeric))] + steps)
        search = GridSearchCV(pipe, grid, cv=cv, scoring=SCORING, refit="f1", n_jobs=-1)
        search.fit(X, y)
        i = search.best_index_
        res = search.cv_results_
        rows.append({
            "modelo": model_name,
            "variables": set_name,
            "parametros": str(search.best_params_),
            **{f"{m}_mean": res[f"mean_test_{m}"][i] for m in SCORING},
            **{f"{m}_std": res[f"std_test_{m}"][i] for m in SCORING},
        })
        print(f"{set_name:15} | {model_name:20} | F1 = {res['mean_test_f1'][i]:.3f}")

con_PageValues  | arbol_sobremuestreo  | F1 = 0.640
con_PageValues  | rf_sobremuestreo     | F1 = 0.669
con_PageValues  | rf_pesos_de_clase    | F1 = 0.676
sin_PageValues  | arbol_sobremuestreo  | F1 = 0.380
sin_PageValues  | rf_sobremuestreo     | F1 = 0.426
sin_PageValues  | rf_pesos_de_clase    | F1 = 0.422


#### Resultados

In [10]:
cv_results = pd.DataFrame(rows).round(4)
cv_results.sort_values("f1_mean", ascending=False)

,modelo,variables,parametros,f1_mean,recall_mean,precision_mean,average_precision_mean,f1_std,recall_std,precision_std,average_precision_std
2,rf_pesos_de_clase,con_PageValues,"{'model__max_depth': None, 'model__min_samples...",0.6758,0.7883,0.5917,0.7216,0.0116,0.0191,0.0124,0.0193
1,rf_sobremuestreo,con_PageValues,"{'model__max_depth': 20, 'model__min_samples_l...",0.6693,0.7870,0.5823,0.7267,0.0147,0.0199,0.0136,0.0168
0,arbol_sobremuestreo,con_PageValues,"{'model__max_depth': 3, 'model__min_samples_le...",0.6396,0.8578,0.5103,0.6412,0.0122,0.0198,0.0153,0.0133
4,rf_sobremuestreo,sin_PageValues,"{'model__max_depth': 20, 'model__min_samples_l...",0.4257,0.5170,0.3619,0.3750,0.0195,0.0154,0.0209,0.0193
5,rf_pesos_de_clase,sin_PageValues,"{'model__max_depth': 10, 'model__min_samples_l...",0.4220,0.6429,0.3142,0.3676,0.0141,0.0219,0.0107,0.0234
3,arbol_sobremuestreo,sin_PageValues,"{'model__max_depth': 5, 'model__min_samples_le...",0.3797,0.7503,0.2554,0.3217,0.0078,0.0675,0.0126,0.0105


#### Guardado de resultados

In [11]:
cv_results.to_csv(RESULTS_DIR / "cv_balanced.csv", index=False)
print(f"Guardado: {RESULTS_DIR / 'cv_balanced.csv'}")

Guardado: results\cv_balanced.csv
